# Разбор соревнования «Тайные клубы школьного чата»

В школе четыре клуба. У нас есть число лайков и дизлайков каждого ученика каждому каналу. Нужно восстановить клубы по реакциям.

Главная идея: **отделить интересы от активности**. Затем учесть, что лайки и дизлайки несут разную информацию.

Пройдём путь от KMeans на сырых лайках до отдельных долей лайков и дизлайков с увеличенным весом дизлайков. Кластеры будем рассматривать через тепловые карты: понижение размерности для этого не требуется.

Для запуска нужны `numpy`, `pandas`, `matplotlib`, `scikit-learn` и `threadpoolctl`. Положите рядом `reactions.csv` и `sample_submission.csv` или запускайте тетрадку в исходной папке комплекта. Положите рядом и `solution.csv` с правильными ответами, чтобы после каждого решения сразу увидеть его ARI. Если файла с ответами нет, тетрадка построит кластеры и сохранит посылку, а расчёт ARI пропустит.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import normalize
from sklearn.metrics import adjusted_rand_score
from threadpoolctl import threadpool_limits

# Ограничение потоков ускоряет небольшие эксперименты на многих компьютерах.
thread_limit = threadpool_limits(limits=1)
RANDOM_STATE = 42

plt.rcParams.update({
    'figure.figsize': (10, 4),
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. Что лежит в таблице

Строка файла — пара «ученик, канал». А объект кластеризации — **ученик**. Поэтому сначала превратим длинную таблицу в две матрицы: «ученик × канал» для лайков и такую же для дизлайков.

Идентификаторы нужны для сопоставления строк и отправки ответа. В признаки их не включаем.

In [ ]:
data_candidates = [Path('reactions.csv'), Path('participant/reactions.csv')]
data_path = next((path for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Положите reactions.csv рядом с тетрадкой.')

reactions = pd.read_csv(data_path)
assert not reactions[['user_id', 'channel_id']].duplicated().any()
assert reactions[['likes', 'dislikes']].notna().all().all()
assert (reactions[['likes', 'dislikes']] >= 0).all().all()
display(reactions.head())
print('Строк:', len(reactions))
print('Учеников:', reactions.user_id.nunique())
print('Каналов:', reactions.channel_id.nunique())

In [ ]:
likes_table = reactions.pivot(index='user_id', columns='channel_id', values='likes')
dislikes_table = reactions.pivot(index='user_id', columns='channel_id', values='dislikes')
dislikes_table = dislikes_table.reindex(index=likes_table.index, columns=likes_table.columns)
assert likes_table.notna().all().all()
assert dislikes_table.notna().all().all()

user_ids = likes_table.index.to_numpy()
channel_ids = likes_table.columns.to_numpy()
likes = likes_table.to_numpy(dtype=float)
dislikes = dislikes_table.to_numpy(dtype=float)

print('Размер каждой матрицы:', likes.shape)
display(likes_table.head())

### Как измеряем качество

**ARI** сравнивает состав групп. Перестановка номеров кластеров ничего не меняет. Значение 1 означает полное совпадение, около 0 соответствует случайному разделению, отрицательные значения тоже возможны.

В каждом блоке сначала строим кластеры по реакциям, затем сверяем их с ответами по `user_id`. Для сравнения шагов используем **ARI на всех учениках**. В итоговой таблице также будут Public и Private. Это результаты одной и той же группировки на разных частях данных. Варианты признаков уже исследовали на этом наборе, поэтому эти части не служат новой независимой проверкой.

In [ ]:
key_candidates = [Path('solution.csv'), Path('private/solution.csv')]
key_path = next((path for path in key_candidates if path.exists()), None)
solution = None
scores = {}

if key_path is not None:
    solution = pd.read_csv(key_path).set_index('user_id')
    assert solution.index.is_unique
    assert set(solution.index) == set(user_ids)
    solution = solution.reindex(user_ids)

def evaluate(name, labels, compare_to=None):
    if solution is None:
        return None
    actual = solution['cluster'].to_numpy()
    scores[name] = {'Все': adjusted_rand_score(actual, labels)}
    if 'Usage' in solution.columns:
        for split in ['Public', 'Private']:
            mask = solution['Usage'].to_numpy() == split
            if mask.any():
                scores[name][split] = adjusted_rand_score(actual[mask], labels[mask])
    score = scores[name]['Все']
    print(f'{name} → ARI {score:.4f}')
    if compare_to is not None:
        delta = score - scores[compare_to]['Все']
        print(f'Изменение относительно «{compare_to}»: {delta:+.4f}')
    return score

## 2. Почему сырые лайки — слабый старт

Два ученика из одного клуба могут оставлять очень разное количество реакций. Например, профили `[2, 8, 0]` и `[20, 80, 0]` выражают одинаковые относительные предпочтения, но евклидово расстояние между ними велико.

KMeans работает с расстояниями между признаковыми векторами. Если использовать сырые счётчики, высокая активность может перевесить различия интересов. Посмотрим на распределение активности и всё же запустим такой бейзлайн.

In [ ]:
activity = (likes + dislikes).sum(axis=1)
print('Реакций на ученика: минимум', int(activity.min()),
      ', медиана', int(np.median(activity)), ', максимум', int(activity.max()))

fig, ax = plt.subplots()
ax.hist(activity, bins=35, color='#5084B5', edgecolor='white')
ax.set(title='Ученики сильно различаются по активности',
       xlabel='Всего реакций ученика', ylabel='Число учеников')
plt.show()

In [ ]:
predictions = {}
feature_sets = {}

def run_kmeans(name, features, n_init=10):
    model = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=n_init)
    labels = model.fit_predict(features)
    predictions[name] = labels
    feature_sets[name] = features
    return labels

baseline_name = 'Сырые лайки'
baseline_labels = run_kmeans(baseline_name, likes)

baseline_summary = pd.DataFrame({
    'Кластер': baseline_labels,
    'Реакций': activity,
}).groupby('Кластер').agg(
    учеников=('Реакций', 'size'),
    медиана_реакций=('Реакций', 'median'),
)
display(baseline_summary)
_ = evaluate(baseline_name, baseline_labels)

Номера кластеров условны. Нас интересует состав групп, а не то, какая получила номер 0. Таблица выше помогает проверить, насколько группы различаются по активности; сама по себе она ещё не измеряет совпадение с настоящими клубами.

На сырых лайках ARI почти нулевой. При этом медианы числа реакций заметно отличаются между кластерами. Дальше попробуем описать, куда ученик направляет свои реакции.

## 3. Добавляем дизлайки и сжимаем счётчики

Храним лайки и дизлайки отдельными блоками. На 12 каналах получаем 24 признака. Преобразование `log(1 + x)` ослабляет влияние крупных счётчиков. Проверим, насколько помогает такое представление.

In [ ]:
both_counts = np.hstack([likes, dislikes])
name = 'Логарифмы реакций'
_ = evaluate(name, run_kmeans(name, np.log1p(both_counts)), compare_to=baseline_name)

## 4. Нормируем длину профиля

Делим каждый вектор на его L2-длину `sqrt(sum(x²))`. Если один ученик в два раза активнее другого, но распределяет реакции по каналам точно так же, после нормировки их профили совпадут.

In [ ]:
name = 'Реакции: L2-нормировка'
_ = evaluate(name, run_kmeans(name, normalize(both_counts)), compare_to='Логарифмы реакций')

## 5. Вычитаем дизлайки и делим на все реакции

Для каждого канала берём `(лайки − дизлайки) / total`, где `total` — число всех реакций ученика. Так знак показывает, каких реакций по этому каналу больше. Деление на `total` позволяет сравнивать активных и менее активных учеников.

In [ ]:
total = np.maximum(activity[:, None], 1)
likes_share = likes / total
dislikes_share = dislikes / total

signed_name = 'Разность реакций / все реакции'
_ = evaluate(signed_name, run_kmeans(signed_name, (likes - dislikes) / total),
         compare_to='Реакции: L2-нормировка')

## 6. Пробуем корни и L2-нормировку

Это отдельная ветка от шага 4. Перед нормировкой берём квадратные корни счётчиков. Например, 1 и 16 превратятся в 1 и 4. Крупные значения меньше растягивают расстояние между профилями.

In [ ]:
sqrt_name = 'Корни реакций + L2'
sqrt_features = normalize(np.sqrt(both_counts))
_ = evaluate(sqrt_name, run_kmeans(sqrt_name, sqrt_features), compare_to='Реакции: L2-нормировка')

## 7. Ещё один подход: уменьшить вес распространённых реакций

Если признак ненулевой почти у всех, сам факт такой реакции слабее различает учеников. Можно попробовать повысить вес более редких признаков. Частоту считаем по всем ученикам отдельно для каждого из 24 столбцов.

В ранее подготовленном авторском решении использовано деление на корень из частоты ненулевых значений, затем L2-нормировка. Это эвристика, а не гарантия: редкий признак может оказаться и шумом. Сравним её с предыдущим решением.

In [ ]:
frequency = (both_counts > 0).mean(axis=0)
frequency = np.maximum(frequency, 0.01)
weighted_counts = both_counts / np.sqrt(frequency)
frequency_features = normalize(weighted_counts)

run_kmeans('Частотные веса + L2', frequency_features, n_init=15)
display(pd.DataFrame({
    'Канал': channel_ids,
    'Доля учеников с лайками': frequency[:len(channel_ids)],
    'Доля учеников с дизлайками': frequency[len(channel_ids):],
}).round(3))
_ = evaluate('Частотные веса + L2', predictions['Частотные веса + L2'], compare_to=sqrt_name)

## 8. Сохраняем обе доли и усиливаем дизлайки

Вернёмся к долям из шага 5. При вычитании один и тот же результат может получиться при нуле реакций и при одинаковом числе лайков и дизлайков. Сохраним два блока отдельно и начнём с одинакового веса.

In [ ]:
equal_name = 'Отдельные доли реакций'
equal_features = np.hstack([likes_share, dislikes_share])
_ = evaluate(equal_name, run_kmeans(equal_name, equal_features), compare_to=signed_name)

Равный вес дал заметный провал. Проверим гипотезу, что дизлайки сильнее различают клубы. Оставим те же доли и удвоим только блок дизлайков. Удвоение признака в четыре раза увеличивает его вклад в квадрат евклидова расстояния. Используем один знаменатель для обоих блоков, число всех реакций ученика.

In [ ]:
DISLIKE_WEIGHT = 2
final_name = 'Доли реакций, вес дизлайков 2'
final_features = np.hstack([likes_share, DISLIKE_WEIGHT * dislikes_share])
final_labels = run_kmeans(final_name, final_features)
_ = evaluate(final_name, final_labels, compare_to=equal_name)

## Сравниваем результаты KMeans

Варианты с частотными весами и с удвоением вклада дизлайков показали близкие результаты. У отдельных долей особенно хорошо виден эффект одного изменения. При равных весах ARI равен 0,1630, а при весе дизлайков 2 достигает 0,9062 на этих данных.

In [ ]:
results = None
if scores:
    results = pd.DataFrame.from_dict(scores, orient='index')
    results.index.name = 'Решение'
    display(results.round(4))

In [ ]:
if results is not None:
    kmeans_results = results.loc[list(feature_sets), 'Все']
    fig, ax = plt.subplots(figsize=(11, 5))
    colors = ['#DB825A' if name == final_name else '#5084B5' for name in kmeans_results.index]
    bars = ax.barh(kmeans_results.index, kmeans_results.values, color=colors)
    ax.bar_label(bars, fmt='%.4f', padding=5)
    ax.invert_yaxis()
    ax.set(xlabel='ARI на всех учениках', xlim=(-0.02, 1.04),
           title='Вклад признаков: один алгоритм, разные представления реакций')
    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)
    fig.tight_layout()
    plt.show()

    print(f"Бейзлайн: {results.loc[baseline_name, 'Все']:.4f}")
    print(f"Отдельные доли с весом 2: {results.loc[final_name, 'Все']:.4f}")

## 9. Смотрим на группы без понижения размерности

У нас всего 12 каналов. Для визуализации достаточно тепловой карты: строки — ученики, столбцы — каналы, цвет — доля реакций. Упорядочим строки по предсказанному кластеру и отдельно покажем средние профили четырёх групп.

Рисунки помогают понять, чем группы различаются. Красивое разделение на рисунке ещё не доказывает совпадение с настоящими клубами.

In [ ]:
order = np.argsort(final_labels, kind='stable')
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
for ax, values, title in zip(
    axes, [likes_share, dislikes_share], ['Доли лайков', 'Доли дизлайков']
):
    image = ax.imshow(values[order], aspect='auto', interpolation='nearest',
                      cmap='YlGnBu', vmin=0, vmax=values.max())
    ax.set_title(title)
    ax.set_xticks(range(len(channel_ids)), channel_ids, rotation=45)
    ax.set_xlabel('Канал')
    fig.colorbar(image, ax=ax, label='Доля всех реакций ученика')
    boundaries = np.flatnonzero(np.diff(final_labels[order])) + 0.5
    for boundary in boundaries:
        ax.axhline(boundary, color='black', linewidth=0.8)
axes[0].set_ylabel('Ученики, сгруппированные по предсказанному клубу')
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for ax, values, title in zip(
    axes, [likes_share, dislikes_share], ['Средняя доля лайков', 'Средняя доля дизлайков']
):
    means = np.vstack([values[final_labels == group].mean(axis=0) for group in range(4)])
    image = ax.imshow(means, aspect='auto', cmap='YlGnBu', vmin=0, vmax=means.max())
    ax.set_title(title)
    ax.set_xticks(range(len(channel_ids)), channel_ids, rotation=45)
    ax.set_yticks(range(4))
    ax.set(xlabel='Канал', ylabel='Предсказанный кластер')
    fig.colorbar(image, ax=ax, label='Средняя доля')
fig.tight_layout()
plt.show()

display(pd.Series(final_labels).value_counts().sort_index().rename('Учеников').to_frame())

## 10. Почему DBSCAN не обязан выигрывать

DBSCAN ищет плотные области и оставляет часть объектов шумом. Нормировка устраняет различия масштаба, но не гарантирует, что четыре клуба станут четырьмя областями с подходящей для DBSCAN плотностью.

Проверим несколько заранее указанных радиусов на тех же нормированных корнях реакций. Для каждого выведем число кластеров и долю шума. Это небольшой опыт, а не исчерпывающий подбор параметров. В ARI все метки `-1` будут считаться одной группой, поэтому множество объектов в шуме может сильно влиять на оценку.

Рядом с количеством групп и долей шума выведем ARI, чтобы сравнить эти варианты с KMeans.

In [ ]:
dbscan_features = feature_sets['Корни реакций + L2']
dbscan_diagnostics = []
for eps in [0.2, 0.3, 0.4, 0.5]:
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(dbscan_features)
    name = f'DBSCAN: eps={eps}'
    predictions[name] = labels
    dbscan_diagnostics.append({
        'Модель': name,
        'Кластеров без шума': len(set(labels) - {-1}),
        'Доля шума': np.mean(labels == -1),
        'ARI': evaluate(name, labels),
    })
display(pd.DataFrame(dbscan_diagnostics).round(3))

## 11. Собираем ответ

Для финального ответа используем отдельные доли реакций с весом дизлайков 2. Предсказания уже получены **без чтения скрытых меток**.

Нужно сохранить соответствие `user_id` и предсказаний. Если рядом есть `sample_submission.csv`, воспроизведём порядок учеников из него. Номер кластера может быть любым целым: перестановка номеров не меняет ARI.

In [ ]:
submission = pd.DataFrame({'user_id': user_ids, 'cluster': final_labels})
sample_path = data_path.parent / 'sample_submission.csv'
if sample_path.exists():
    sample = pd.read_csv(sample_path)
    assert sample.user_id.is_unique
    assert set(sample.user_id) == set(submission.user_id)
    submission = sample[['user_id']].merge(submission, on='user_id', validate='one_to_one')

assert submission.user_id.is_unique
assert len(submission) == len(user_ids)
assert submission.notna().all().all()
assert pd.api.types.is_integer_dtype(submission.cluster)
submission.to_csv('clubs_submission.csv', index=False)
display(submission.head())
print('Сохранён clubs_submission.csv:', len(submission), 'строк')

## Что важно вынести из решения

1. Сначала выбираем объект кластеризации. Здесь это ученик, поэтому длинную таблицу реакций разворачиваем в признаки ученика.
2. Разная активность мешает сравнивать интересы по сырым счётчикам. Нормировка и преобразование счётчиков меняют смысл расстояния.
3. Дизлайки стоит исследовать отдельно. Вычитание реакций и два отдельных блока признаков — разные представления данных.
4. Большая часть улучшения здесь получена подготовкой признаков для обычного KMeans. Нормировка сама по себе не гарантирует хорошего результата DBSCAN.
5. Тепловые карты позволяют увидеть профили групп без понижения размерности.

Попробуйте поменять вес дизлайков или `random_state` и сравнить устойчивость групп. Скрытый ключ после соревнования подходит для учебного сравнения, но многократный подбор по нему делает полученную оценку оптимистичной.